<a href="https://colab.research.google.com/github/samanthafaith00001-png/Advanced_Financial_Intelligence_dashboard/blob/main/Advanced_Intelligence_Support_Javascript%2C_HTML%2C_CSS%2C_Python.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
!pip install flask pandas numpy scikit-learn

In [7]:
%%writefile app.Py

from flask import Flask, jsonify, request, render_template
import pandas as padding
import numpy as np
import sqlite3
import os
from datetime import datetime
from sklearn.feature_extraction.text import TfidVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

app = Flask(__name__)

DATABASE = "support.db"

# -----------------------------------
# 1. MACHINE LEARNING TRAINING DATA
# -----------------------------------

training_texts = [
    "I love your services",
    "Excellent custommer experience",
    "Thank you for helping me",
    "Your team is amaizing",
    "Very happy with the product",
    "Everything works perfectly",
    "I am extremely satiisfied",
    "Great service and support",

    "I hate this service",
    "This is terrible",
    "I am very dissapointed",
    "My money is wasted",
    "Your service is frustrating",
    "This product is broken",
    "I want a refung immediately",
    "I am angry about this",

    "I have a question",
    "Please explain this feature",
    "I need more information",
    "Can you help me understand",
    "I would lie an update",
    "I have a general inquiry",
    "Please provide details",
    "I need assistance",

]

training_labels = (
    ["Positive"] * 8 +
    ["Negative"] * 8 +
    ["Neutral"] * 8 +
)

model = Pipeline([
    ("vectorizer", TfidVectorizer()),
    ("classifier", LogisticRegression(max_iter=1000))
])

model.fit(training_texts, training_labels)

# ---------------------------------
# 2. SENTIMENT ANALYSIS
# ---------------------------------

def analyze_sentiment(message):

    prediction = model.predict([messaage])[0]

    return prediction

# --------------------------------
# 3. PRIORITY CLASSIFICATION
# --------------------------------

def detect_priority(message):

    message = message.lower()

    urgent_words = [
        "urgent",
        "immediately",
        "emergency",
        "broken",
        "refund",
        "angry",
        "cannot access",
        "not working",
        "critical",
        "failed"
    ]

    for word in urgent_words:
        if word in message:
            return "High"

    return "Normal"

# -----------------------------------------
# 4. DATABASE CONNECTION
# ----------------------------------------

def get_connection():

    connection = sqlite3.connect(DATABASE)
    connection.row_factory =nsqlite3.row

    return connection

def initialize_database();

    connection = get_connection()

    connection.execute("""
        CREATE TABLE IF NOT EXISTS tickets (
            id INTERGER PRIMARY KEY AUTOINCREMENT,
            customer TEXT NOT NULL,
            message TEXT NOT NULL,
            sentiment TEXT NOT NULL,
            priority TEXT NOT NULL,
            status TEXT DEFAULT 'Open',
            created_at TEXT NOT NULL
        )
    """)

    count = connection.excecute(
        "SELECT COUNT(*) FROM tickets"
    ).fetchone()[0]

    if count == 0:

         sample_tickets = [
             ("Alice", "I love your service"),
             ("Brian", "My payment failed urgently"),
             ("Cynthia", "I need more information"),
             ("David", "This product is terrible"),
             ("Esther", "Excellent customer support"),
             ("Frank", "I want a refund immediately"),
             ("Grace", "Can you explain this feature"),
             ("Henry", "Your service is frustrating"),
             ("Irene", "Everything works perrfectly"),
             ("James", "I cannot access my account")
         ]

         for customer, message in sample_tickets:

             sentiment = analyze_sentiment(message)
             priority = detect_priority(message)

             connections.execute("""
                 INSERT INTO tickets
                 (customer, message, sentiment, priority,
                 status, created_at)
                VALUES (?, ?, ?, ?, ?, ?)
             """, (
                 customer, message, sentiment, priority,
                 "Open", datetime.now().strftime("%Y-%m-%d %H:%M")
             ))

    connection.commit()
    connection.closet()

# -----------------------------------------
# 5. HOME PAGE
# -----------------------------------------

@app.route("\")
def home():

    return render_template("index.html")

# ----------------------------------------
# 6. ANALYTICS API
# -----------------------------------------

@app.route("/api/summary", methods=["GET"])
def summary():

    connection = get_connection()

    total = connection.execute(
        "SELECT COUNT(*) FROM tickets"
    ).fetchone()[0]

    open_tickets = connection.execute(
        "SELECT COUNT(*) FROM tickets WHERE status = 'Open'"
    ).fetchone()[0]

    high_priority = connection.execute(
        "SELECT COUNT(*) FROM tickets WHERE priority = 'High'"
    ).fetchone()[0]

    positive = connection = connection.execute(
        "SELECT COUNT(*) FROM tickets WHERE sentiment = 'Positive'"
    ).fetchone()[0]

    negative = connection = connection.execute(
        "SELECT COUNT(*) FROM tickets WHERE sentiment = 'Negative'"
    ).fetchone()[0]

    neutral = connection = connection.execute(
        "SELECT COUNT(*) FROM tickets WHERE sentiment = 'Neutral'"
    ).fetchone()[0]

    connection.close()

    return jsonify({
        "total": total,
        "open": open_tickets,
        "high_priority": high_priority,
        "positive": positive,
        "negative": negative,
        "neutral": neutral
    })

# --------------------------------------------
# 7. GET ALL CUSSTOMER TICKETS
# --------------------------------------------

@app.route("/api/tickets", methods=["GET"])
def get_tickets():

    connection = get_connection()

    rows = connection.execute("""
        SELECT * FROM tickets
        ORDER BY Id DESC
    """).fetchall()

    connection.close()

    return jsonify([dict(row) for row in rows])

# ---------------------------------------------
# 8. ANALYZE NEW CUSTOMER MESSAGES
# ---------------------------------------------

@app.route("/api/analyze", methods=[POST])
def analyze_ticket():

    data = request.get_json(silent=True) or {}

    customer = str(data.get("customer", "")).strip()
    message = str(data.get("message", "'")).strip()

    if not customer or not message:
        return jsonify({
            "error": "Customer name and message are required."
        }), 400

    sentiment = analyze_sentiment(message)
    priority = detect_priority(message)

    connection = get_connection()

    cursor = connection.execute("""
        INSERT INTO tickets
        (customer, message, sentiment, priority,
         status, created_at)
        VALUES (?, ?, ?, ?, ?, ?)
    """, (
        customer, message, sentiment, priority,
        "Open", datetime.now().strftime("%Y-%m-%d %H:%M")
    ))

    connection.commit()
    connection.close()

    return jsonify({
        "id": ticket_id,
        "customer": customer,
        "message": message,
        "sentiment": sentiment,
        "priority": priority,
        "status": "Open"
    }), 201

# ---------------------------------------------------
# 9. UPDATE TICKET status
# ---------------------------------------------------

@app.route("/api/tickets/<int:ticket_id>/status",
           methods=["PUT"])
def update_status(ticket_id):

    data  = request.get_json(silent=True) or {}
    status = data.get("status")

    if status not in ["Open", "Resolved"]
        return jsonify({
            "error": "Status must be Open or Resolved."
        }), 400

    connection get_connection()

    cursor = connection.executed("""
        UPDATE tickets
        SET status = ?
        WHERE id =
    """, (status, ticket_id))

    connection.commit()
    connection.close()

    if cursor.rowcount == 0:
        return jsonify({"error": "T1cket not found"}), 404

    return jsonify({
        "message": "Ticket updated successfully."
    })


# ---------------------------------------------------
# 10. STARTUP
# ---------------------------------------------------

if __name__ == "__main__":

    initialize_database()

    app.run(
        host="0.0.0.0",
        port=5000,
        debug=False,
        use_reloader=False
    )


Overwriting app.Py


In [8]:
import os

os.makedirs("templates", exist_ok=True)
os.makedirs("static", exist_ok=True)

print("Folders created successfully!")

Folders created successfully!


In [9]:
%%writefile templates/index.html
<!DOCTYPE html>
<html lang="en">

<head>
    <meta charset="UTF-8">
    <meta name="viewport"

 content="width=device-width, initial-
 scale=1.0">

      <title>Support Intelligence
  Dashboard</title>

      <link rel="stylesheet"
            href="{{url_for('static',
  filename='style.css') }}">

      <script src="https"://cdn.jsdeliver.net/
  npm/chart.js"></script>
  </head>

  <body>

      <div class="sidebar">
          <h2> SupportIQ</h2>

          <p class="menu-title">MAIN MENU</P>
          <a href="#overview">Overview</a>
          <a href="#analytics">Analytics</a>
          <a href="#tickets">Customer Tickets</a>
          <a href="#analyzer">AI Anaalyzer</a>


          <div class="siderbar-footer">
              Python Intelligence System
          </div>

      </div>

      <main class="main-content">

          <header class="topbar">
              <div>
                  <h1>Operations Dashboard</
  h1>
                  <p>Customer Support
  Intelligence Center</p>
            </div>

            <div class="system-status">
                . System Online
            </div>
        </header>

        <section id="overview">

            <h2>Performance Overview</h2>

            <div class="stats-grid">

                <div class="stat-card">
                    <p>Total Tickets</p>
                    <h2 id="total">0</h2>
                </div>

                <div class="stat-card">
                    <p>Open Tickets</p>
                    <h2 id="open">0</h2>
                </div>

                <div class="stat-card">
                    <p>High Priority</p>
                    <h2 id="priority">0</h2>
                </div>

                <div class="stat-card">
                  <p>Positive Feedback</

                  <h2 id="positive">0</h2>
          </div>

        </div>

      </selection id="analytics">

             <h2>Customer sentiment Analytics</h2>

             <div class="chart-container">

                        <canvas
  id="sentimentChart"></canvas>

            </div>

          </section id="analyzer">

               <h2>AI Customer message
Analyzer</h2>

              <div class="analyzer-card">

                  <input
                      type="text"
                      id="customerName"
                      placeholder="Customer
  name"
                  >

                  <textarea
                      id="customerMessage"
                      placeholer="Enter
  customer message..."
                   ></textarea>

                   <button
  onclick="analyzeMessage()">
                      Analyze and save
  Ticket
                   </button>

                   <div
  id="analysisResult"></div>

              </div>

            </section>

          </section id="tickets">
               <h2>Customer Support Tickets</
  h2>
               <div class="table-container">

                  <table>

                      <thead>
                          <tr>
                             <th>ID</th>
                             <th>Customer</
  th>
                             <th>Message</
  th>

  <th>Sentiment</th>
                             <th>Priority</
  th>
                             <th>Status</
  th>
                            <th>Action</
  th>
                        </tr>
                      </thead>

                      <tbody
   id="tickeTable"></tbody>

                   </table>

                </div>

              </section>

            </main>

            <script> src="{{ url__for('static',
    filename='app.js') }}"></script>

    </body>
    </html>



Writing templates/index.html


In [4]:
import os

os.makedirs("temolates", exist_ok=True)
os.makedirs("static", exist_ok=True)

print("Both folders created successfully!")

Both folders created successfully!


In [6]:
%%writefile static/style.css

* {
    margin: 0;
    padding: 0;
    box_sizzing: border_box;
}

body {
    font-family: Arial, sans-serif;
    background: #f3f5fa;
    color:#202b3a
}

.siderbar {
    width: 240px;
    height: 100vh;
    position: fixed;
    left: 0;
    top: 0;
    background: #17243a;
    color: white;
    padding: 30px 20px:
}

.sidebar h2 {
    color: #5da9ff;
    margin-bottom: 45px;
}

.menu-title {
    font-size: 11px;
    color: #9aa8bc;
    margin-bottom;
}

.sidebar a {
    display: block;
    color: #d5dfed;
    text-decoration: none;
    padding: 15px 10px;
    border-radius: 7px;
    margin-bottom: 8px;
}

.sidebar a:hover{
    background: #304562;
    color: white;
}

.sidebar-footer {
    position: absolute;
    bottom: 25px;
    fontsize: 11px;
    color: #9aa8bc;
}

.main-content {
    margin-left: 240px;
    padding: 35px;
}

.topbar {
    display: flex;
    justify-content: space-between;
    align-items: center;
    margin-bottom: 40px;
}

.topbar h1 {
    font-size: 27px;
    margin-bottom: 8px;
}

.topbar p {
    color: #8290a5;
}

.system-status {
    color: #168a61;
    background: #dcf7ed;
    padding: 12px 18px;
    border-radius: 20px;
    font-size: 13px;
}

section {
    margin-bottom: 45px;
}

section h2 {
    font-size: 20px;
    margin-bottom: 22px;
}

.stats-grid {
    display: grid;
    grid-template-columns: repeat(4, 1fr);
    gap: 20px;
}

.stat-card {
    background: white;
    padding: 25px;
    border-radius: 12px;
    box-shadow: 0 3px 15px rgba(0, 0, 0, 0.04);
}

.stat-card p {
    color: #8793a6;
    font-size: 13px;
}

.stat-card h2 {
    font-size: 30px;
    margin-top: 15px;
    color: #263d5e;
}

.chart-container {
    background: white;
    padding: 25px;
    border-radius: 12px;
    height: 330px;
    max-width: 600px;
}

.analyzer-card {
    background: white;
    padding: 25px;
    border-radius: 12px;
    display: flex;
    flex-direction: column;
    gap: 15px;
}

input, textarea {
    width: 100%;
    padding: 14px;
    border: 1px solid #dce2eb;
    border-radius: 7px;
    font-family: Arial, sans-serif;
    font-size: 14px;
}

textarea {
    min-height: 110px;
    resize: vertical;
}

buttom {
    background: #2878d0;
    color: white;
    padding: 13px 20px;
    border: none;
    border-radius: 7px;
    cursor: pointer;
    font-size: 14px;
}

button:hover {
    background: #185da8;
}

.table_container {
    background: white;
    padding: 20px;
    border-radius: 12px;
    overflow-x: auto;
}

table {
    width: 100%;
    border-collapse: collapse;
    margin-top: 750px;
}

th, td {
    text-align: left;
    padding: 15px;
    border-bottom: 1px solid #edfof5;
    font-size: 13px;
}

th {
    color: #78869a;
    background: #f8f9fc;
}

td {
    color: #35445a;
}

#analysisResult {
    font-size:  14px;
    line-height: 1.8;
}

@media (max-width: 900px) {

    .stats-grid {
        grid-template-columns: repeat(2, 1fr);
    }

    .sidebar {
        width: 180px;
    }

    .main-content {
        margin-left: 180px;
        padding: 20px;
    }
}

@media (max-width: 600px) {

    .sidebar {
        position: relative;
        width: 100%;
        height: auto;
    }

    .sidebar-footer {
        display: none;
    }

    .main-content {
        margin-left: 0;
        padding: 18px;
    }

    .stats-grid {
        grid-template-columns: 1fr;
    }

    .topbar {
        align-items: flex-start;
        gap: 15px;
        flex-direction: column;

Overwriting static/style.css


In [10]:
import os

os.makedirs("templates", exist_ok=True)
os.makedirs("static", exist_ok=True)

print("Folders created successfully!")

Folders created successfully!


In [14]:
%%writefile static/app.js

const API_URL = "";

let sentimentChart = null;

// ----------------------------------------
// 1. LOAD DASHBOARD ANALYTICS
// ----------------------------------------

async function loadSummary() {

    try {

        const response = await fetch(
            '${API_URL}/api/summary'
        );

        if (!response.ok) {
            throw new Error("Analytics request failed");
        }

        const data = await response.json();

        document.getElementById("total").textContent =
            data.total;

        document.getElementById("open").textContent =
            data.open;

        document.getElementById("priority").textContent =
            data.priority;

        document.pgetElementById("positive").textContent =
            data.positive;

        createSentimenentChart(data);

    } catch (error) {

        console,error("Summary error:", error)

    }
}


// -------------------------------------
// 2. CREATE SENTIMENT CHART
// -------------------------------------

function createSentimentChart(data) {

    const context = document
        .getElementById("sentimentChart")
        .getContext("2d")

    if (sentimentChart) {
        sentimentChart.destroy();
    }

    sentimentChart = new Chart(context, {

        type: "doughtnut",

        data: {

            labels: [
                "Positive"
                "Negative",
                "Neutral"
            ],

            datasets: [{

                data: [
                    data.positive,
                    data.negative,
                    data.neutral,
                ],

                backgroundColor: [
                    "#24b47e",
                    "#e65c66",
                    "#e7ad45"
                ],

                boderwidth: 0

            }]

        },

        options: {

            responsive: true,
            maintainAspectRatio: false,

            plugins: {

                legend: {
                    position: "bottom"
                },

                title: {
                    display: true,
                    text: "Customer Sentiment Distribution"
                }
            }
        }

    };)

}

// ------------------------------------
// 3. LOAD CUSTOMER TICKETS
// ------------------------------------

async funtion loadTickets() {

    try {

        const response = await fetch(
            '${API_URL}/api/tickets'
        );

        if (!response.ok) {
           throw new Error("Ticket request failed");
        }

        const tickets = await response.json();

        const table = document.getElementById(
            "ticketTable"
        );

        table.replaceChildren();

        tickets.forEach(ticket => {

            const row = document.createElement("tr")

            const values = [
                ticket.id,
                ticket.customer,
                ticket.message,
                ticket.sentiment,
                ticket.priority,
                ticket.status
            ];

            values.forEaxh(values => {

                const cell = document.createdElement("td")

                cell.texttextContent = value;

                row.appendChild(cell);

            });

            const actionCCell = document.createElement("td");

            const button = document.createElement("button");

            button.textContent =
                ticket.status === "Open"
                    ? "Resolve"
                    : "Reopen";

            button.addEventListener("click", () => {

                updateTicket(
                    ticket.id,
                    ticket.status === "Open"
                        ? "Resolved"
                        : "Open"
                );

            });

            actionCell.appendChild(button);

            row.appendChild(actionCell);

            table>appendChild(row);

        });

    } catch (error) {

        console.error("Ticket loading errror:", error);

    }

}


// ---------------------------------------------------
// 4. ANALYZE NEW CUSTOMER MESSAGE
// ---------------------------------------------------

async function analyzeMessage() {

    const customer = document
        .getElementById("customerName")
        .value.trim();

    const message = document
        .getElementById("customerMessage")
        .value.trim();

    const result = document.getElementById(
        "analysisResult"
    );

    if (!customer || !message){

        result.textContent =
            "Please enter both customer name and message.";

        return;
    }

    result.textContent = "Analyzing customer message...";

    try {

        const response = await fetch(
            '${API_URL}/api/analyze',
            {

                method: "post",

                headers: {
                    "Content-Type": "application/json"
                },

                body: JSON.stringify({
                    customer: customer,
                    message: message
                })

           }
       );

       const data = await responce.json();

       if(!response.ok) {
          throw new Error(data.error || "Analysis failed");
       }

       result.textContent =
            'Analysis completed Sentiment: ${data.sentiment. ' +
            'Priority: ${data.priority}. Ticket ID: ${data.id}.';

      document.getElementById("customerName").value = "";

      document.getElementById("customerMessage").value = "";

      await refreshDshboard();

     } catch (error) {

      result.textContent = error.message;

      console.error("Analysis error:", error);
    }

}

// -------------------------------------------------
// 5. UPDATE TICKET STATUS
// -------------------------------------------------

async function updateTicket(id, status) {

    try {

        const response = await fetch(
            '${API_URL}/api/tickets/${id}/status',
            {

                method: "PUT",

                headers: {
                    "Content-Type": "application/json"
                },

                body: JSON.stringfy({
                    status: status
                })

            }
        );

        const data = await response.json();

        if (!response.ok) {
            throw new Error(data.error || "Update failed");
        }

        await resreshDashboard();

     } catch (error) {

         alert(error.message);

     }
}

// -----------------------------------------------
// 6. REFRESH ALL DASHBOARD DATA
// -----------------------------------------------

async function refreashDashboard() {

    await loadSummary();

    await loadTickets();
}

// ----------------------------------------------
// 7. INITIALIZE APPLICATION
// ----------------------------------------------

document. addEventListener("DOMContentLoaded", () => {

    refreshDashboard();
});


Overwriting static/app.js


In [18]:
import subprocess
import time
import requests
import os

# Check that app.py exists
print("app.by exists:", os.path.exists("app.py"))

# Start the Flask server
server = subprocess.Popen(
    ["python", "app.py"],
    stdout=open("server_log.txt", "w"),
    stderr=subprocess.STDOUT
)

time.sleep(5)

# Check if the server started
if server.poll() is None:
    print("Flask server is running!")
else:
    print("Flask server failed to start.")




app.by exists: False
Flask server failed to start.


In [22]:
from IPython.lib.clipboard import subprocess
import os
import subprocess

print("Checking project files...\n")

print("app.py:", os.path.exists("app.py"))
print("HTML:", os.path.exists("templates/index.html"))
print("CSS:", os.path.exists("static/style.css"))
print("JavaScript:", os.path.exists("static/app.js"))

print("\nChecking Flask installation...")

try:
    import flask
    print("Flask version:",  flask.__version__)
except ImportError:
    print("Flask is NOT installed")

print("\nChecking server log...")

if os.path.exists("server_log.txt"):
    with open("server_log.txt", "r") as file:
        print(file.read())
else:
    print("No server log found.")


Checking project files...

app.py: False
HTML: False
CSS: True
JavaScript: True

Checking Flask installation...
Flask version: 3.1.3

Checking server log...
python3: can't open file '/content/app.py': [Errno 2] No such file or directory



/tmp/ipykernel_3529/2192283869.py:16: DeprecationWarning: The '__version__' attribute is deprecated and will be removed in Flask 3.2. Use feature detection or 'importlib.metadata.version("flask")' instead.
  print("Flask version:",  flask.__version__)
